In [4]:
! pip install contextily
! pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 2.1 MB/s eta 0:00:00


In [14]:
import folium
import geopandas as gpd
from geopy.distance import geodesic
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pyproj import Transformer
from shapely.geometry import box, Point, LineString
import matplotlib.pyplot as plt
import contextily as ctx
import heapq
import math
import time
import networkx as nx
import osmnx as ox
import matplotlib
import IPython.display as display
# matplotlib.use('TkAgg')  # Or 'Qt5Agg' / 'GTK3Agg' depending on what's installed


class MeetingPoint:

    def __init__(self):
        self.gdf = gpd.read_file('/content/sample_data/IND_adm2.shp')
        self.gdf = self.gdf.to_crs(epsg=4326)

        self.roads_gdf = gpd.read_file('/content/sample_data/IND_roads.shp')
        crs_epsg = 32644
        # Project to metric CRS for accurate distance measurement
        if self.roads_gdf.crs != f'epsg:{crs_epsg}':
            self.roads_gdf = self.roads_gdf.to_crs(epsg=crs_epsg)

    def prepare_road_graph(self):
        road_graph = nx.Graph()
        # Build graph from road segments
        for _, row in self.roads_gdf.iterrows():
            if isinstance(row.geometry, LineString):
                start_point = row.geometry.coords[0]
                end_point = row.geometry.coords[-1]
                length = row.geometry.length  # length in meters
                # Use coordinates as node IDs or modify based on your data
                road_graph.add_edge(start_point, end_point, weight=length)
        return road_graph

    @staticmethod
    def get_road_distance(city_1, city_2):
        # Get coordinates using Nominatim (built-in in osmnx)
        location_1 = ox.geocode(city_1)
        location_2 = ox.geocode(city_2)

        # Download the driving road network around the two locations
        road_graph = ox.graph_from_point(location_1, dist=50000, network_type='drive')  # 50 km buffer

        # Get the nearest nodes in the graph to the two cities
        node1 = ox.distance.nearest_nodes(road_g, location_1[1], location_1[0])  # (lon, lat)
        node2 = ox.distance.nearest_nodes(road_g, location_2[1], location_2[0])

        # Compute the shortest path length in meters (Dijkstra by default)
        distance = nx.shortest_path_length(road_graph, node1, node2, weight='length')

        # Convert meters to kilometers
        return distance / 1000

    def get_city_coordinates(self, city_name, city_column='city'):
        # Load shapefile
        gdf = self.roads_gdf

        # Optional: print column names to verify
        print("Available Columns:", gdf.columns)

        # Filter by city name (adjust 'city_column' as per your shapefile)
        city_row = gdf[gdf[city_column].str.contains(city_name, case=False, na=False)]

        if not city_row.empty:
            # Assuming Point geometry
            point = city_row.iloc[0].geometry
            if isinstance(point, Point):
                return point.x, point.y
            else:
                # If LineString, get the first point of the line
                return city_name, point.coords[0]
        else:
            print(f"City '{city_name}' not found.")
            return city_name, None

    def get_geo_data(self, city):
        city_data = self.gdf[self.gdf['NAME_2'].str.lower() == city.lower()]
        if city_data.empty:
            print(f"City {city} not found in the GeoDataFrame.")
        return city_data

    def get_distance(self, city_1, city_2):
        city_1_data = self.get_geo_data(city_1)
        city_2_data = self.get_geo_data(city_2)

        if city_1_data.empty:
            print(f"City {city_1} not found in the GeoDataFrame.")
            return
        if city_2_data.empty:
            print(f"City {city_2} not found in the GeoDataFrame.")
            return

        city1_geom = city1_data.centroid.iloc[0]
        city2_geom = city2_data.centroid.iloc[0]
        city1_lat, city1_lon = city1_geom.y, city1_geom.x
        city2_lat, city2_lon = city2_geom.y, city2_geom.x

        distance = self.get_distance_between_points(city1_lat, city1_lon, city2_lat, city2_lon)
        print(f"Distance between {city1} and {city2} is {distance:.2f} km.")

    def find_edges(self, city1, city2):
        print(f"City 1: {city1} & City 2: {city2}")
        city1_data = self.get_geo_data(city1)
        city2_data = self.get_geo_data(city2)

        if city1_data.empty:
            print(f"City {city1} not found in the GeoDataFrame.")
            return
        if city2_data.empty:
            print(f"City {city2} not found in the GeoDataFrame.")
            return

        # Re-project to a projected CRS (e.g., EPSG:3857) before calculating centroids
        city1_data = city1_data.to_crs(epsg=3857)
        city2_data = city2_data.to_crs(epsg=3857)

        city1_geom = city1_data.centroid.iloc[0]
        city2_geom = city2_data.centroid.iloc[0]

        # Re-project the centroids back to EPSG:4326
        city1_geom = gpd.GeoSeries([city1_geom], crs="EPSG:3857").to_crs(epsg=4326).iloc[0]
        city2_geom = gpd.GeoSeries([city2_geom], crs="EPSG:3857").to_crs(epsg=4326).iloc[0]

        city1_lat, city1_lon = city1_geom.y, city1_geom.x
        city2_lat, city2_lon = city2_geom.y, city2_geom.x

        print(f"City 1: {city1_lat}, {city1_lon}")
        print(f"City 2: {city2_lat}, {city2_lon}")
        print(f"City 1: {city1} & City 2: {city2} are {self.get_distance_between_points(city1_lat, city1_lon, city2_lat, city2_lon):.2f} km apart.")
        range_buffer_gdx, _ = self.create_buffer((city1_lat, city1_lon), (city2_lat, city2_lon))
        radius_to_neighbouring_city = self.get_distance_between_points(city1_lat, city1_lon, city2_lat, city2_lon)
        city1_neighbours=self.get_neighbouring_city(range_buffer_gdx, (city1, (city1_lat, city1_lon)), radius_to_neighbouring_city)
        city2_neighbours=self.get_neighbouring_city(range_buffer_gdx, (city2, (city2_lat, city2_lon)), radius_to_neighbouring_city)
        city1_edges=self.get_edges(city1,city1_neighbours)
        city2_edges = self.get_edges(city2, city2_neighbours)
        return city1_edges, city2_edges

    # def get_all_edges(self, city1, city2):



    def get_edges(self, city, city_neighbours, is_print_enabled=True):
        if is_print_enabled:
            for neighbour_city in city_neighbours:
                print(f"City : {city} --{neighbour_city[1]}-->{neighbour_city[0]}")

        final_city_edges = []
        for neighbour_city in city_neighbours:
            final_city_edges.append((city, neighbour_city[0], neighbour_city[1], neighbour_city[2]))

        return final_city_edges

    def get_all_neighbours(self, city1_lat, city1_lon,city2_lat, city2_lon):
        range_buffer_gdx, _ = self.create_buffer((city1_lat, city1_lon), (city2_lat, city2_lon))
        # Find neighboring cities within the range buffer
        neighboring_cities = self.gdf[self.gdf.geometry.centroid.within(range_buffer_gdx.geometry.iloc[0])]

        return neighboring_cities

    def get_neighbouring_city(self, range_buffer_gdx, city, distance=100):


        # Find neighboring cities within the range buffer
        neighboring_cities = self.gdf[self.gdf.geometry.centroid.within(range_buffer_gdx.geometry.iloc[0])]

        all_cities_box=self.prepare_neighbours(city, neighboring_cities)
        neighboring_cities_within_range=self.get_neighboring_cities(city, distance)

        print(f"Neighboring cities within the range is : {len(all_cities_box)}")
        print(f"Neighboring cities within the range is : {len(neighboring_cities_within_range)}")
        final_neighboring_cities = self.get_intersection(all_cities_box,neighboring_cities_within_range)
        print(f"Neighboring cities within the range is : {len(final_neighboring_cities)}")
        return final_neighboring_cities


    def get_intersection(self, list1, list2):
        # Create sets of the first elements
        set1 = set([x[0] for x in list1])
        set2 = set([x[0] for x in list2])

        # Get common first elements
        common_keys = set1 & set2

        # Filter from list1 or list2 based on the common first elements
        intersection = [item for item in list1 if item[0] in common_keys]
        return intersection


    def prepare_neighbours(self, city, neighboring_cities):
        city_list = []
        for index, row in neighboring_cities.iterrows():
            city_name = row['NAME_2']
            city_geom = row.geometry.centroid
            city_lat, city_lon = city_geom.y, city_geom.x
            city_distance = self.get_distance_between_points(city[1][0], city[1][1], city_lat, city_lon)
            # print(f"City: {city_name}, Distance: {city_distance:.2f} km")
            city_list.append((city_name, city_distance, (city_lat, city_lon)))
        return city_list

    def convert_from_lat_lon_to_xy(self, lat, lon):
        # Create GeoDataFrame in EPSG:4326
        gdf = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs="EPSG:4326")
        # Convert to EPSG:3857
        gdf_3857 = gdf.to_crs(epsg=3857)
        # Extract the converted coordinates
        x, y = gdf_3857.geometry.iloc[0].x, gdf_3857.geometry.iloc[0].y
        return (x, y)

    def get_neighboring_cities(self, city, radius_km, plot=False):
        """
        city: (city_name, (lat, lon))
        radius_km: Radius in kilometers
        plot: Boolean to control plotting
        """
        lat, lon = city[1]

        # Convert city center to x, y in meters (EPSG:3857)
        city_x, city_y = self.convert_from_lat_lon_to_xy(lat, lon)

        # Convert radius to meters
        radius_m = radius_km * 1000

        # Create GeoDataFrame for city center in EPSG:3857
        city_point = gpd.GeoDataFrame(geometry=[Point(city_x, city_y)], crs="EPSG:3857")

        # Project the cities GeoDataFrame to EPSG:3857 for accurate distance calculations
        cities_projected = self.gdf.to_crs(epsg=3857)

        # Create circular buffer around the center city
        circular_buffer = city_point.buffer(radius_m).iloc[0]

        # Filter neighboring cities within the buffer
        neighboring_cities = cities_projected[cities_projected.geometry.within(circular_buffer)]

        # Plotting
        if plot:
            fig, ax = plt.subplots(figsize=(10, 10))

            # Plot the country boundary for context
            self.gdf.to_crs(epsg=3857).boundary.plot(ax=ax, color='gray', linewidth=0.5)

            # Plot circular buffer
            gpd.GeoSeries([circular_buffer], crs="EPSG:3857").boundary.plot(ax=ax, color='red', linewidth=2,
                                                                            label='Search Radius')

            # Plot center city point
            city_point.plot(ax=ax, color='blue', markersize=100, label=f'Center: {city[0]}')

            # Plot neighboring cities
            if not neighboring_cities.empty:
                neighboring_cities.plot(ax=ax, color='green', markersize=50, label='Neighboring Cities')

            # Optional: Add basemap
            try:
                # print(ctx.providers)
                ctx.add_basemap(ax, crs='EPSG:3857', source=ctx.providers.OpenStreetMap.Mapnik)
            except Exception as e:
                print("Basemap loading skipped:", e)

            # Custom legend
            legend_elements = [
                Line2D([0], [0], marker='o', color='w', label=f'Center: {city[0]}',
                       markerfacecolor='blue', markersize=10),
                Patch(facecolor='green', edgecolor='green', label='Neighboring Cities'),
                Patch(facecolor='none', edgecolor='red', label='Search Radius')
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            plt.show()

        # Return neighbor info (if needed)
        return self.prepare_neighbours(city, neighboring_cities)


    def get_distance_between_points(self, lat1, lon1, lat2, lon2):
        # Example coordinates (lat, lon)

        is_proj1 = self.is_projected((lat1, lon1))
        is_proj2 = self.is_projected((lat2, lon2))

        # Convert projected to lat/lon
        if is_proj1 and not is_proj2:
            lat1, lon1 = self.convert_from_xy_to_latlon(lon1, lat1)
        elif is_proj2 and not is_proj1:
            lat2, lon2 = self.convert_from_xy_to_latlon(lon2, lat2)
        elif is_proj1 and is_proj2:
            # Optional: convert both if needed, or raise warning
            lat1, lon1 = self.convert_from_xy_to_latlon(lon1, lat1)
            lat2, lon2 = self.convert_from_xy_to_latlon(lon2, lat2)

        # Now both are in lat/lon - Safe to compute distance
        return geodesic((lat1, lon1), (lat2, lon2)).kilometers

    def is_projected(self, cords):
        # Rough heuristic: if lat > 90 or lon > 180, it's likely projected
        return abs(cords[0]) > 180 or abs(cords[1]) > 180

    def convert_from_xy_to_latlon(self, x, y):
        transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(x, y)
        return (lat, lon)

    def create_buffer(self, coord1, coord2, delta=0.1):
        """
        coord1, coord2: (latitude, longitude) tuples
        delta: buffer amount added to the bounding box (in degrees)
        """
        # Unpack coordinates
        lat1, lon1 = coord1
        lat2, lon2 = coord2

        # Create the bounding rectangle (min_lon, min_lat, max_lon, max_lat) with delta buffer
        min_lon, max_lon = min(lon1, lon2) - delta, max(lon1, lon2) + delta
        min_lat, max_lat = min(lat1, lat2) - delta, max(lat1, lat2) + delta

        # Create the rectangular box with buffer
        rect = box(min_lon, min_lat, max_lon, max_lat)

        # Convert to GeoDataFrame
        rect_gdf = gpd.GeoDataFrame(geometry=[rect], crs="EPSG:4326")

        # Optional: Points GeoDataFrame for plotting
        points_gdf = gpd.GeoDataFrame(geometry=[Point(lon1, lat1), Point(lon2, lat2)], crs="EPSG:4326")

        return rect_gdf, points_gdf



    def greedy_best_first_search(self,city1,city2):
        start = (city1, city2)
        frontier = []
        heapq.heappush(frontier, (self.heuristic(city1, city2), start, 0, [], [], []))

        visited = set()
        nodes_generated = 0
        max_frontier = 1
        start_time = time.perf_counter()

        while frontier:
            max_frontier = max(max_frontier, len(frontier))
            h, (your_city, friend_city), cost_so_far, path, my_trv_path, friend_trv_path = heapq.heappop(frontier)
            state = (your_city, friend_city)

            if state in visited:
                continue
            visited.add(state)

            if your_city == friend_city:
                duration = time.perf_counter() - start_time
                return your_city, cost_so_far, nodes_generated, max_frontier, duration, path + [your_city], my_trv_path + [your_city], friend_trv_path + [your_city]

            for you_next in G.neighbors(your_city):
                for friend_next in G.neighbors(friend_city):
                    you_cost = G[your_city][you_next]['weight']
                    friend_cost = G[friend_city][friend_next]['weight']
                    next_cost = cost_so_far + max(2 * you_cost, 2 * friend_cost)
                    h_next = self.heuristic(you_next, friend_next)
                    # if h_next is None:
                    #     h_next = float('inf')  # Fallback in case heuristic still returns None
                    nodes_generated += 1
                    heapq.heappush(frontier,
                                   (h_next, (you_next, friend_next), next_cost, path + [f"{your_city}-{friend_city}"], my_trv_path + [f"{your_city}"], friend_trv_path + [f"{friend_city}"]))
                    # print(f"Your City : {your_city} & Friend City : {friend_city}")

        return None

    def a_star_search(self,city1,city2):
        start = (city1, city2)
        frontier = []
        heapq.heappush(frontier, (self.heuristic(city1, city2), start, 0, [], [], []))

        visited = set()
        nodes_generated = 0
        max_frontier = 1
        start_time = time.perf_counter()

        while frontier:
            max_frontier = max(max_frontier, len(frontier))
            f, (your_city, friend_city), cost_so_far, path, my_trv_path, friend_trv_path = heapq.heappop(frontier)
            state = (your_city, friend_city)

            if state in visited:
                continue
            visited.add(state)

            if your_city == friend_city:
                duration = time.perf_counter() - start_time
                return your_city, cost_so_far, nodes_generated, max_frontier, duration, path + [your_city], my_trv_path + [your_city], friend_trv_path+ [your_city]

            for you_next in G.neighbors(your_city):
                for friend_next in G.neighbors(friend_city):
                    you_cost = G[your_city][you_next]['weight']
                    friend_cost = G[friend_city][friend_next]['weight']
                    step_cost = max(2 * you_cost, 2 * friend_cost)
                    g_next = cost_so_far + step_cost
                    h_next = self.heuristic(you_next, friend_next)
                    f_next = g_next + h_next
                    nodes_generated += 1
                    heapq.heappush(frontier,
                                   (f_next, (you_next, friend_next), g_next, path + [f"{your_city}-{friend_city}"], my_trv_path + [f"{your_city}"], friend_trv_path + [f"{friend_city}"]))

        return None

    def heuristic(self, city_1, city_2):
        try:
            coord1 = city_cords[city_1]
            coord2 = city_cords[city_2]
            return self.euclidean_distance(coord1, coord2)
        except KeyError:
            print(f"Missing coordinates for {city1} or {city2}")
            return float('inf')

    # def heuristic(self, source_city, target_city):
    #     # Compute shortest path distance using Dijkstra's algorithm
    #     try:
    #         return self.get_road_distance(source_city, target_city)
    #     except nx.NetworkXNoPath:
    #         print(f"No road path between {source_city} and {target_city}")
    #         return None


    def euclidean_distance(self, coord1, coord2):
        lat1, lon1 = coord1
        lat2, lon2 = coord2
        return math.sqrt((lat1 - lat2) ** 2 + (lon1 - lon2) ** 2)

    def plot_graph(self):
        pos = {city: (coord[1], coord[0]) for city, coord in city_cords.items()}
        plt.figure(figsize=(12, 8))
        # nx.draw(G, pos, with_labels=True, node_size=800, node_color='skyblue', font_size=10, edge_color='gray')
        nx.draw(G.subgraph(pos.keys()), pos, with_labels=True, node_size=800,
                node_color='skyblue', font_size=10, edge_color='gray')
        labels = nx.get_edge_attributes(G, 'weight')
        filtered_labels = {(u, v): d for (u, v), d in labels.items() if u in pos and v in pos}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=filtered_labels)
        # nx.draw_networkx_edge_labels(G, pos, edge_labels=labels)
        plt.title("City Map with Edges as Distances")
        plt.show()

    def plot_graph_goal(self, G, goal_city):
        """
        Plots the graph with:
        - Transition cost (g) on the edges
        - Heuristic (h) values near the nodes
        """
        plt.figure(figsize=(12, 8))
        pos = nx.spring_layout(G, seed=42)  # You can also use self.pos if you have coordinates

        # Compute heuristic h(n) as shortest distance to the goal city
        heuristics = {}
        for node in G.nodes():
            try:
                heuristics[node] = nx.shortest_path_length(G, node, goal_city, weight='weight')
            except nx.NetworkXNoPath:
                heuristics[node] = float('inf')

        # Draw nodes
        nx.draw(G, pos, with_labels=True, node_color='skyblue', node_size=2000, font_size=10)

        # Draw edges with transition cost (g)
        edge_labels = {(u, v): f"g={d['weight']}" for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=9)

        # Draw heuristic values (h) near each node
        for node, (x, y) in pos.items():
            plt.text(x, y + 0.08, f"h={heuristics[node]:.1f}", fontsize=9, color='green', ha='center')

        plt.title("City Map with Transition Cost (g) and Heuristic (h)")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    def plot_cities(self, city1_edges, city2_edges, gbfs_meeting_city, a_star_meeting_city):
        fig, ax = plt.subplots(figsize=(10, 10))
        self.gdf.boundary.plot(ax=ax, linewidth=0.5, edgecolor='green')

        city_coords = {}
        for u, v, w, loc in city1_edges + city2_edges:
            city_coords[u] = loc
            city_coords[v] = loc

        # Select a subset of cities to plot
        cities_to_plot = {city1_edges[0][0], city2_edges[0][0], gbfs_meeting_city, a_star_meeting_city}
        additional_cities = list(city_coords.keys() - cities_to_plot)[:4]
        cities_to_plot.update(additional_cities)

        for city in cities_to_plot:
            lat, lon = city_coords[city]
            ax.plot(lon, lat, marker='o', color='blue', markersize=5)
            ax.text(lon, lat, city, fontsize=8)

        gbfs_lat, gbfs_lon = city_coords[gbfs_meeting_city]
        a_star_lat, a_star_lon = city_coords[a_star_meeting_city]

        ax.plot(gbfs_lon, gbfs_lat, marker='o', color='red', markersize=10, label=f'GBFS Meeting City {gbfs_meeting_city}')
        ax.plot(a_star_lon, a_star_lat, marker='o', color='green', markersize=10, label=f'A* Meeting City {a_star_meeting_city}')

        ax.legend()
        plt.show()


    def get_all_edges(self):
        # Collect all unique cities from city1_edges and city2_edges
        all_cities = set()
        for u, v, w, loc in city1_edges + city2_edges:
            all_cities.add(u)
            all_cities.add(v)

        # Find edges for all cities
        all_edges = []
        for city in all_cities:
            city_data = main.get_geo_data(city)
            if not city_data.empty:
                city_geom = city_data.to_crs(epsg=3857).centroid.iloc[0]
                city_geom = gpd.GeoSeries([city_geom], crs="EPSG:3857").to_crs(epsg=4326).iloc[0]
                city_lat, city_lon = city_geom.y, city_geom.x

                # Find neighboring cities within a certain distance
                neighboring_cities = main.get_neighboring_cities((city, (city_lat, city_lon)), radius_km=150)

                for neighbor_city, distance, (neighbor_lat, neighbor_lon) in neighboring_cities:
                    all_edges.append((city, neighbor_city,distance, (neighbor_lat, neighbor_lon)))

        return list(set(all_edges))

    def is_road_connected(self, city1_geom, city2_geom, roads_gdf, buffer_distance_m=50000):
        # Ensure projected CRS
        roads_gdf = roads_gdf.to_crs(epsg=3857)
        city1_geom = city1_geom.to_crs(epsg=3857).geometry.iloc[0]
        city2_geom = city2_geom.to_crs(epsg=3857).geometry.iloc[0]

        # Buffer in meters
        buffer1 = city1_geom.buffer(buffer_distance_m)
        buffer2 = city2_geom.buffer(buffer_distance_m)

        # Check for intersection
        for _, road in roads_gdf.iterrows():
            if road.geometry.intersects(buffer1) and road.geometry.intersects(buffer2):
                return True
        return False

    def plot_result_on_map(self, my_path, friend_path, meeting_city, city_cords):
        """
        Plot GBFS paths for both players and meeting point on Folium map.
        """
        # Center map roughly on the meeting point
        meeting_coords = city_cords.get(meeting_city)
        print(f"Meeting points: {meeting_coords}")
        if self.is_projected(meeting_coords):
            meeting_coords = (self.convert_from_xy_to_latlon(meeting_coords[0], meeting_coords[1]))
            print(f"Meeting points: {meeting_coords}")
        m = folium.Map(location=(meeting_coords[0], meeting_coords[1]), zoom_start=6)

        # Plot My Path
        my_points = []
        for city in my_path:
            if city in city_cords:
                lon, lat = city_cords[city]
                if self.is_projected((lon, lat)):
                    lat, lon = self.convert_from_xy_to_latlon(lon, lat)
                my_points.append((lon, lat))
                folium.Marker(
                    location=(lon, lat),
                    popup=f"My City: {city}",
                    icon=folium.Icon(color='blue')
                ).add_to(m)

        if my_points:
            folium.PolyLine(my_points, color='blue', weight=3, tooltip='My Path').add_to(m)

        # Plot Friend Path
        friend_points = []
        for city in friend_path:
            if city in city_cords:
                lon, lat = city_cords[city]
                if self.is_projected((lon, lat)):
                    lat, lon = self.convert_from_xy_to_latlon(lon, lat)
                friend_points.append((lon, lat))
                folium.Marker(
                    location=(lon, lat),
                    popup=f"Friend City: {city}",
                    icon=folium.Icon(color='green')
                ).add_to(m)

        if friend_points:
            folium.PolyLine(friend_points, color='green', weight=3, tooltip="Friend's Path").add_to(m)

        # Plot Meeting City
        if meeting_city in city_cords:
            lon, lat = city_cords[meeting_city]
            folium.Marker(
                location=(lon, lat),
                popup=f"Meeting City: {meeting_city}",
                icon=folium.Icon(color='red', icon='star')
            ).add_to(m)

        return m

main = MeetingPoint()
city1='Delhi'
city2='Kolkata'
city1_edges, city2_edges = main.find_edges(city1, city2)
is_road_data_enabled = True
if is_road_data_enabled:
    all_edges_within_region=main.get_all_edges()
else:
    all_edges_within_region = city1_edges + city2_edges
print(city1_edges)
print(city2_edges)
print(all_edges_within_region)
city_distances = []
# Graph connections for both
G = nx.Graph()
all_edges = []

if is_road_data_enabled:
    road_g=main.prepare_road_graph()
    G = road_g
    # for u, v, w, loc in all_edges_within_region:
    #     if u != v:
    #         is_road_connected = main.is_road_connected(main.get_geo_data(u), main.get_geo_data(v),
    #                                                    main.roads_gdf)
    #         if is_road_connected:
    #             all_edges.append((u,v,w,loc))
    # all_edges_within_region=all_edges

for u, v, w, loc in all_edges_within_region:
    if u != v:
        G.add_edge(u, v, weight=w)
        # G.add_node(u)
        city_distances.append((v,loc))

for u, v, w, loc in city1_edges + city2_edges:
    city_distances.append((v,loc))

city_cords = dict(city_distances)
print(city_cords)

# city_cords_roads_list = []
# for city in city_cords.items():
#     city_cords_roads_list.append((city[0],main.get_city_coordinates(city[0])))
#
# city_cords_roads = dict(city_cords_roads_list)

# main.plot_graph()

print("\nGBFS Result")
result = main.greedy_best_first_search(city1,city2)
# gbfs_meeting_city = result[0]
if result:
    meet, cost, nodes, space, duration, path, my_trv_path, friend_trv_path = result
    print(f" My City City: {city1} & Friend City: {city2}")
    print(f" Meeting City: {meet}")
    print(f"Total Cost: {cost}")
    print(f"Nodes Generated: {nodes}")
    print(f"Max Frontier Size: {space}")
    print(f"Execution Time: {duration:.4f} sec")
    # print(f"Path: {path}")
    print(f"My Path: {my_trv_path}")
    print(f"Friend Path: {friend_trv_path}")
    # main.plot_graph_goal(G,meet)
    # main.plot_cities(city1_edges, city2_edges, meet, None)
    # Plot the map
    m = main.plot_result_on_map(my_trv_path, friend_trv_path, meet, city_cords)

    # Save and display
    display.display(m)

print("\nA* Search Result")
result = main.a_star_search(city1,city2)
if result:
    meet, cost, nodes, space, duration, path, my_trv_path, friend_trv_path = result
    print(f" My City City: {city1} & Friend City: {city2}")
    print(f" Meeting City: {meet}")
    print(f"Total Cost: {cost}")
    print(f"Nodes Generated: {nodes}")
    print(f"Max Frontier Size: {space}")
    print(f"Execution Time: {duration:.4f} sec")
    # print(f"Path: {path}")
    print(f"My Path: {my_trv_path}")
    print(f"Friend Path: {friend_trv_path}")
    # main.plot_graph_goal(G, meet)
    # main.plot_cities(city1_edges, city2_edges, None, meet)
    # Plot the map
    m = main.plot_result_on_map(my_trv_path, friend_trv_path, meet, city_cords)

    # Save and display
    display.display(m)


City 1: Delhi & City 2: Kolkata
City 1: 28.64668524464709, 77.10896714468635
City 2: 22.551741094616265, 88.35234106515814
City 1: Delhi & City 2: Kolkata are 1314.83 km apart.


<ipython-input-14-a788b2c241c1>:173: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  neighboring_cities = self.gdf[self.gdf.geometry.centroid.within(range_buffer_gdx.geometry.iloc[0])]


Neighboring cities within the range is : 169
Neighboring cities within the range is : 366
Neighboring cities within the range is : 158


<ipython-input-14-a788b2c241c1>:173: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  neighboring_cities = self.gdf[self.gdf.geometry.centroid.within(range_buffer_gdx.geometry.iloc[0])]


Neighboring cities within the range is : 169
Neighboring cities within the range is : 310
Neighboring cities within the range is : 154
City : Delhi --1047.5069103264307-->Araria
City : Delhi --841.8397792358543-->Aurangabad
City : Delhi --1054.327905291446-->Banka
City : Delhi --959.569145921951-->Begusarai
City : Delhi --761.4358856706418-->Bhabua
City : Delhi --1057.6763411650525-->Bhagalpur
City : Delhi --816.2428399524774-->Bhojpur
City : Delhi --775.7311884352783-->Buxar
City : Delhi --924.4973498986666-->Darbhanga
City : Delhi --892.5048940191996-->Gaya
City : Delhi --757.207887449433-->Gopalganj
City : Delhi --1008.1677876347452-->Jamui
City : Delhi --863.0972873454097-->Jehanabad
City : Delhi --1098.8473186520794-->Katihar
City : Delhi --1002.6376826290889-->Khagaria
City : Delhi --1100.2613412451935-->Kishanganj
City : Delhi --975.3205896367533-->Lakhisarai
City : Delhi --1017.4786451082002-->Madhepura
City : Delhi --932.6025036157054-->Madhubani
City : Delhi --1009.4319975291


A* Search Result
 My City City: Delhi & Friend City: Kolkata
 Meeting City: Azamgarh
Total Cost: 1487.128540431294
Nodes Generated: 394200
Max Frontier Size: 149120
Execution Time: 7.1895 sec
My Path: ['Delhi', 'Gautam Buddha Nagar', 'Aligarh', 'Hathras', 'Mainpuri', 'Kannauj', 'Unnao', 'Bara Banki', 'Faizabad', 'Ambedkar Nagar', 'Azamgarh']
Friend Path: ['Kolkata', 'Hugli', 'Barddhaman', 'Birbhum', 'Deoghar', 'Jamui', 'Nalanda', 'Bhojpur', 'Ballia', 'Mau', 'Azamgarh']
Meeting points: (26.055238356369475, 83.04119512160075)
